In [1]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

I0000 00:00:1790489136.992619   22296 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790489137.540611   22296 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790489140.322103   22296 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical"
testing_path = "../Datasets/data/test/Medical"
valid_path = "../Datasets/data/vaild/Medical"

In [3]:
train_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

In [4]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical",
    shuffle = False
)

Found 63746 images belonging to 3 classes.
Found 8529 images belonging to 3 classes.
Found 11997 images belonging to 3 classes.


In [5]:
training_set.class_indices

{'CT': 0, 'MRI': 1, 'X-Ray': 2}

In [6]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))

I0000 00:00:1790489148.191161   22296 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [7]:
base_model.trainable = False

In [8]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(3,activation="sigmoid")(x)

In [9]:
model = Model(inputs = base_model.input,outputs = output)

In [10]:
model.compile(optimizer=Adam(learning_rate=0.002),metrics=["accuracy"],loss="categorical_crossentropy")

In [11]:
early_stop = EarlyStopping(monitor="loss",restore_best_weights=True)

In [12]:
model.fit(training_set,validation_data=(valid_set),epochs=4,callbacks=[early_stop])

Epoch 1/4


I0000 00:00:1790489151.498598   22296 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1790489158.181612   22475 service.cc:153] XLA service 0x7de040037690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790489158.181665   22475 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1790489158.509704   22475 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790489160.520129   22475 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1790489161.134575   22475 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_19123__.271
I0000 00:00:1790489181.350741   22475 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3515/3985 ━━━━━━━━━━━━━━━━━━━━ 1:15 162ms/step - accuracy: 0.9884 - loss: 0.0336

I0000 00:00:1790489752.023650   22477 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_19123__.271


3985/3985 ━━━━━━━━━━━━━━━━━━━━ 741s 178ms/step - accuracy: 0.9965 - loss: 0.0139 - val_accuracy: 0.9988 - val_loss: 0.0042
Epoch 2/4
3985/3985 ━━━━━━━━━━━━━━━━━━━━ 618s 155ms/step - accuracy: 0.9989 - loss: 0.0074 - val_accuracy: 0.9996 - val_loss: 0.0034
Epoch 3/4
3985/3985 ━━━━━━━━━━━━━━━━━━━━ 613s 154ms/step - accuracy: 0.9981 - loss: 0.0056 - val_accuracy: 1.0000 - val_loss: 1.9945e-07
Epoch 4/4
3985/3985 ━━━━━━━━━━━━━━━━━━━━ 617s 155ms/step - accuracy: 0.9925 - loss: 0.0080 - val_accuracy: 0.9996 - val_loss: 8.4217e-04


In [13]:
prediction = model.predict(testing_set)

750/750 ━━━━━━━━━━━━━━━━━━━━ 69s 84ms/step


In [14]:
true_prediction = np.argmax(prediction,axis=1)

In [15]:
true_prediction

array([0, 0, 0, ..., 2, 2, 2], shape=(11997,))

In [16]:
y_true = testing_set.classes

In [17]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3989
           1       1.00      1.00      1.00       781
           2       1.00      1.00      1.00      7227

    accuracy                           1.00     11997
   macro avg       1.00      1.00      1.00     11997
weighted avg       1.00      1.00      1.00     11997



array([[3989,    0,    0],
       [   0,  781,    0],
       [   0,    0, 7227]])

In [19]:
model.save("types_of_medical_model.keras")